# 09 — Single Image Compare Mínimo

Notebook mínimo para validar el pipeline de explicabilidad sobre **una sola imagen local**.

Este notebook no implementa lógica pesada: llama a los scripts y módulos ya creados.

Flujo:

```text
imagen local
→ generación best-of-N
→ generated_ids
→ post-softmax cross-attention
→ QK logits
→ Grad-CAM
→ métricas espaciales
→ figura comparativa
```

Usarlo como prueba local antes de escalar al notebook final de 25 imágenes × 3 modelos.


## 0. Instrucciones rápidas

Guardar este notebook en:

```text
image-captioning/notebooks/09_single_image_compare_minimo.ipynb
```

Abrir Jupyter desde la raíz del repo:

```bash
cd ~/Documents/Vision\ Artificial/tp_final_vision/image-captioning
jupyter notebook
```

Luego abrir este notebook desde la carpeta `notebooks/` y ejecutar las celdas en orden.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown

def find_repo_root(start: Path | None = None) -> Path:
    """Busca la raíz del repo subiendo desde el cwd hasta encontrar src/ y models/."""
    start = (start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src").exists() and (p / "models").exists():
            return p
    raise RuntimeError(
        "No pude encontrar la raíz del repo. "
        "Abrí Jupyter desde image-captioning/ o ajustá ROOT manualmente."
    )

ROOT = find_repo_root()
print("ROOT:", ROOT)


## 1. Configuración

La imagen por defecto es `data/img_prueba/prueba1.jpeg`, que ya fue probada.

También podés cambiarla por:

```python
IMAGE_PATH = ROOT / "data/img_prueba/perro.jpg"
```


In [ ]:
IMAGE_PATH = ROOT / "data/img_prueba/prueba1.jpeg"
MODEL_DIR = ROOT / "models/blip_base"

OUT_DIR = ROOT / "outputs/notebook_minimo_single_image/prueba1_base_gradcam"
FIG_DIR = OUT_DIR / "figures"

DEVICE = "cpu"
SEEDS = [42]
MAX_NEW_TOKENS = 12
SKIP_GRADCAM = False

print("IMAGE_PATH:", IMAGE_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("OUT_DIR:", OUT_DIR)

assert IMAGE_PATH.exists(), f"No existe IMAGE_PATH: {IMAGE_PATH}"
assert MODEL_DIR.exists(), f"No existe MODEL_DIR: {MODEL_DIR}"


## 2. Mostrar imagen original

In [ ]:
img = Image.open(IMAGE_PATH).convert("RGB")
print("size:", img.size)
print("mode:", img.mode)

display_width = 360
display_height = int(display_width * img.size[1] / img.size[0])
display(img.resize((display_width, display_height)))


## 3. Ejecutar pipeline completo

Esta celda llama a:

```text
scripts/run_single_image_compare.py
```

Si ya existen outputs y no querés recomputar, podés saltearla manualmente.


In [ ]:
cmd = [
    sys.executable,
    str(ROOT / "scripts/run_single_image_compare.py"),
    "--image-path", str(IMAGE_PATH),
    "--model-dir", str(MODEL_DIR),
    "--output-dir", str(OUT_DIR),
    "--device", DEVICE,
    "--seeds", *map(str, SEEDS),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
]

if SKIP_GRADCAM:
    cmd.append("--skip-gradcam")

print("Comando:")
print(" ".join(cmd))

subprocess.run(cmd, cwd=ROOT, check=True)


## 4. Cargar caption generada

In [ ]:
caption_path = OUT_DIR / "caption.json"
assert caption_path.exists(), f"No existe: {caption_path}"

with open(caption_path, "r", encoding="utf-8") as f:
    caption_data = json.load(f)

caption = caption_data.get("caption")
chosen_seed = caption_data.get("chosen_seed")
score = caption_data.get("score")
token_ids = caption_data.get("token_ids", [])

display(Markdown(f"**Caption elegida:** {caption}"))
print("chosen_seed:", chosen_seed)
print("score:", score)
print("n_token_ids:", len(token_ids))


## 5. Inspeccionar métricas espaciales

In [ ]:
metrics_csv = OUT_DIR / "spatial_metrics_rows.csv"
assert metrics_csv.exists(), f"No existe: {metrics_csv}"

df_metrics = pd.read_csv(metrics_csv)
df_metrics


## 6. Inspeccionar shapes de heatmaps

In [ ]:
npz_path = OUT_DIR / "compare_heatmaps.npz"
assert npz_path.exists(), f"No existe: {npz_path}"

z = np.load(npz_path, allow_pickle=True)

for k in z.files:
    arr = z[k]
    print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")


## 7. Generar figura comparativa

Esta celda llama a:

```text
scripts/plot_single_image_compare.py
```

No vuelve a correr el modelo. Solo usa el `.npz` ya generado.


In [ ]:
cmd = [
    sys.executable,
    str(ROOT / "scripts/plot_single_image_compare.py"),
    "--image-path", str(IMAGE_PATH),
    "--npz-path", str(npz_path),
    "--output-dir", str(FIG_DIR),
    "--max-tokens", "8",
    "--alpha", "0.45",
]

print("Comando:")
print(" ".join(cmd))

subprocess.run(cmd, cwd=ROOT, check=True)


## 8. Mostrar figura final

In [ ]:
fig_path = FIG_DIR / "single_image_heatmap_grid.png"
assert fig_path.exists(), f"No existe figura: {fig_path}"

display(Markdown(f"**Figura generada:** `{fig_path.relative_to(ROOT)}`"))

fig = Image.open(fig_path)
print("figure size:", fig.size)
display(fig)


## 9. Interpretación mínima

Cosas a observar:

- si `post_softmax` es más plano o repetitivo entre tokens;
- si `qk_logits` tiene más contraste espacial;
- si `gradcam` apunta a regiones distintas;
- si las métricas reflejan esa divergencia.

Este notebook es la versión mínima local. El notebook final debería escalar este flujo a:

```text
25 imágenes × 3 modelos × 3 métodos
```


## 10. Próximo paso

Cuando este notebook corra completo sin errores, el siguiente paso es crear el notebook final:

```text
notebooks/07_explicabilidad_comparada.ipynb
```

Ese notebook va a reutilizar el mismo flujo, pero con loop sobre imágenes/modelos y layout final en:

```text
outputs/notebook_comparativo/
```
